<a href="https://colab.research.google.com/github/kharasyn7963-byte/STAT-520/blob/main/STAT%20520%20Project%20Parts%206%20and%207%20-%20Jaden.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install lifelines statsmodels

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4118 sha256=f214911b8e53cd64bb116e0eea8d0ecd928ae14095fe24d3d620ac983952ed5c
  Stored in directory: /root/.cache/pip/wheels/7e/16/46/9477f188924292d3bf1fb8fb42844201591abfc19b7ba6d868
Successfully built autograd-gamma
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [lifelines]


In [2]:
import pandas as pd
import numpy as np

from scipy import stats
from statsmodels.stats.proportion import (proportions_ztest, confint_proportions_2indep)
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

In [4]:
df = pd.read_csv("glioma_cleaned_eda.csv")

# Makes code easier to read
df = df.rename(columns={ "age_at_initial_pathologic_diagnosis": "age", "OS.time": "time", "OS": "event"})

df.head()

,bcr_patient_barcode,type,age,gender,race,histological_type,histological_grade,vital_status,tumor_status,event,time,OS_months
0,TCGA-02-0001,GBM,44.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,358.0,11.760841
1,TCGA-02-0003,GBM,50.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,144.0,4.730618
2,TCGA-02-0004,GBM,59.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,345.0,11.333771
3,TCGA-02-0006,GBM,56.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,558.0,18.331143
4,TCGA-02-0007,GBM,40.0,FEMALE,WHITE,Treated primary GBM,NaN,Dead,WITH TUMOR,1.0,705.0,23.160315


In [5]:
# GBM = 1 means GBM and GBM = 0 means LGG
df["GBM"] = (df["type"] == "GBM").astype(int)

# Male = 1 means male and Male = 0 means Female
df["Male"] = (df["gender"] == "MALE").astype(int)

df.head()

,bcr_patient_barcode,type,age,gender,race,histological_type,histological_grade,vital_status,tumor_status,event,time,OS_months,GBM,Male
0,TCGA-02-0001,GBM,44.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,358.0,11.760841,1,0
1,TCGA-02-0003,GBM,50.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,144.0,4.730618,1,1
2,TCGA-02-0004,GBM,59.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,345.0,11.333771,1,1
3,TCGA-02-0006,GBM,56.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,558.0,18.331143,1,0
4,TCGA-02-0007,GBM,40.0,FEMALE,WHITE,Treated primary GBM,NaN,Dead,WITH TUMOR,1.0,705.0,23.160315,1,0


In [8]:
# Race simplified to White vs. Non-White
df["NonWhite"] = np.where( df["race"].isna(), np.nan, (df["race"] != "WHITE").astype(int))

df.head(50)

,bcr_patient_barcode,type,age,gender,race,histological_type,histological_grade,vital_status,tumor_status,event,time,OS_months,GBM,Male,NonWhite
0,TCGA-02-0001,GBM,44.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,358.0,11.760841,1,0,0.0
1,TCGA-02-0003,GBM,50.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,144.0,4.730618,1,1,0.0
2,TCGA-02-0004,GBM,59.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,345.0,11.333771,1,1,0.0
3,TCGA-02-0006,GBM,56.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,558.0,18.331143,1,0,0.0
4,TCGA-02-0007,GBM,40.0,FEMALE,WHITE,Treated primary GBM,NaN,Dead,WITH TUMOR,1.0,705.0,23.160315,1,0,0.0
5,TCGA-02-0009,GBM,61.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,322.0,10.578187,1,0,0.0
6,TCGA-02-0010,GBM,20.0,FEMALE,WHITE,Treated primary GBM,NaN,Dead,WITH TUMOR,1.0,1077.0,35.381078,1,0,0.0
7,TCGA-02-0011,GBM,18.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,630.0,20.696452,1,0,0.0
8,TCGA-02-0014,GBM,25.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,2512.0,82.522996,1,1,0.0
9,TCGA-02-0015,GBM,50.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,627.0,20.597898,1,1,0.0


In [9]:
# Represents a ten year increase in age, centered around the average age
# Centering makes the GBM coefficient easier to interpret because it compares GBM and LGG patients at approximately the average age instead of at age zero
df["age_centered10"] = (df["age"] - df["age"].mean()) / 10

df.head()

,bcr_patient_barcode,type,age,gender,race,histological_type,histological_grade,vital_status,tumor_status,event,time,OS_months,GBM,Male,NonWhite,age_centered10
0,TCGA-02-0001,GBM,44.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,358.0,11.760841,1,0,0.0,-0.692799
1,TCGA-02-0003,GBM,50.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,144.0,4.730618,1,1,0.0,-0.092799
2,TCGA-02-0004,GBM,59.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,345.0,11.333771,1,1,0.0,0.807201
3,TCGA-02-0006,GBM,56.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,558.0,18.331143,1,0,0.0,0.507201
4,TCGA-02-0007,GBM,40.0,FEMALE,WHITE,Treated primary GBM,NaN,Dead,WITH TUMOR,1.0,705.0,23.160315,1,0,0.0,-1.092799


In [12]:
# The test is asking whether the population mean age differs between the two tumor groups
# Welch's version was used because it doesn't require equal variances

lgg_age = df.loc[df["type"] == "LGG", "age"].dropna()
gbm_age = df.loc[df["type"] == "GBM", "age"].dropna()

t_stat, p_value = stats.ttest_ind(lgg_age, gbm_age, equal_var=False)

difference = gbm_age.mean() - lgg_age.mean()

print("LGG mean:", f"{lgg_age.mean():.3f}")
print("GBM mean:", f"{gbm_age.mean():.3f}")
print("GBM minus LGG:", f"{difference:.3f}")
print("t-statistic:", f"{t_stat:.3f}")
print("p-value:", p_value)

LGG mean: 42.942
GBM mean: 57.829
GBM minus LGG: 14.887
t-statistic: -17.862
p-value: 7.026316686312552e-63


In [16]:
# This test examines whether gender and tumor type are independent

gender_table = pd.crosstab(df["type"], df["gender"])

chi2, chi_p, degrees_freedom, expected = stats.chi2_contingency(gender_table, correction=False)

cramers_v = np.sqrt(chi2 / gender_table.to_numpy().sum())

# p-value is slightly below 0.05 however, Cramer's V is very small meaning the actual association is weak

print(gender_table)
print("Chi-square:", f"{chi2:.3f}")
print("Degrees of freedom:", degrees_freedom)
print("p-value:", f"{chi_p:.3f}")
print("Cramer's V:", f"{cramers_v:.3f}")
print("Expected counts:")
print(expected)

gender  FEMALE  MALE
type                
GBM        230   366
LGG        230   285
Chi-square: 4.195
Degrees of freedom: 1
p-value: 0.041
Cramer's V: 0.061
Expected counts:
[[246.76867687 349.23132313]
 [213.23132313 301.76867687]]


In [18]:
# creates a binary variable for death within a year

def determine_one_year_death(row):
    if pd.isna(row["time"]):
        return np.nan

    if row["event"] == 1 and row["time"] <= 365:
        return 1

    if row["time"] >= 365:
        return 0

    return np.nan


df["death_within_1_year"] = df.apply(determine_one_year_death,axis=1)

df.head()

,bcr_patient_barcode,type,age,gender,race,histological_type,histological_grade,vital_status,tumor_status,event,time,OS_months,GBM,Male,NonWhite,age_centered10,death_within_1_year
0,TCGA-02-0001,GBM,44.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,358.0,11.760841,1,0,0.0,-0.692799,1.0
1,TCGA-02-0003,GBM,50.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,144.0,4.730618,1,1,0.0,-0.092799,1.0
2,TCGA-02-0004,GBM,59.0,MALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,345.0,11.333771,1,1,0.0,0.807201,1.0
3,TCGA-02-0006,GBM,56.0,FEMALE,WHITE,Untreated primary (de novo) GBM,NaN,Dead,WITH TUMOR,1.0,558.0,18.331143,1,0,0.0,0.507201,0.0
4,TCGA-02-0007,GBM,40.0,FEMALE,WHITE,Treated primary GBM,NaN,Dead,WITH TUMOR,1.0,705.0,23.160315,1,0,0.0,-1.092799,0.0


In [19]:
one_year = df.dropna(subset=["death_within_1_year"])

mortality_table = pd.crosstab(
    one_year["type"],
    one_year["death_within_1_year"]
)

print(mortality_table)

death_within_1_year  0.0  1.0
type                         
GBM                  298  232
LGG                  394   26


In [27]:
# This tests if probability of mortality is different between GBM and LGG

gbm_deaths = mortality_table.loc["GBM", 1]
gbm_total = mortality_table.loc["GBM"].sum()

lgg_deaths = mortality_table.loc["LGG", 1]
lgg_total = mortality_table.loc["LGG"].sum()

z_stat, z_p = proportions_ztest([gbm_deaths, lgg_deaths],[gbm_total, lgg_total])

gbm_rate = gbm_deaths / gbm_total
lgg_rate = lgg_deaths / lgg_total

risk_difference = gbm_rate - lgg_rate

risk_ci = confint_proportions_2indep(
    gbm_deaths,
    gbm_total,
    lgg_deaths,
    lgg_total,
    method="wald"
)

print("GBM mortality:", f"{gbm_rate:.3f}")
print("LGG mortality:", f"{lgg_rate:.3f}")
print("Risk difference:", f"{risk_difference:.3f}")
print("95% CI:", risk_ci)
print("z-statistic:", f"{z_stat:.3f}")
print("p-value:", z_p)

GBM mortality: 0.438
LGG mortality: 0.062
Risk difference: 0.376
95% CI: (np.float64(0.32771601294548536), np.float64(0.4239461613581984))
z-statistic: 12.935
p-value: 2.8698223613486677e-38


In [28]:
# drops 36 patients (35 missing race and 1 missing survival time)

model_data = df.dropna(
    subset=[
        "time",
        "event",
        "age_centered10",
        "GBM",
        "Male",
        "NonWhite"
    ]
).copy()

model_data["age_GBM_interaction"] = (
    model_data["age_centered10"] *
    model_data["GBM"]
)

In [29]:
# The Cox model estimates hazard ratios
# A hazard ratio above 1 means a higher instantaneous risk of death, while a hazard ratio below 1 means a lower instantaneous risk

model_columns = [
    "time",
    "event",
    "age_centered10",
    "GBM",
    "Male",
    "NonWhite",
    "age_GBM_interaction"
]

cph = CoxPHFitter()

cph.fit(
    model_data[model_columns],
    duration_col="time",
    event_col="event",
    formula=(
        "age_centered10 + GBM + Male + "
        "NonWhite + age_GBM_interaction"
    )
)

cph.print_summary()

<lifelines.CoxPHFitter: fitted with 1075 total observations, 480 right-censored observations>
             duration col = 'time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 1075
number of events observed = 595
   partial log-likelihood = -3322.02
         time fit was run = 2026-11-25 04:43:49 UTC

---
                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                            
age_centered10       0.52      1.69      0.07            0.39            0.66                1.47                1.94
GBM                  1.52      4.56      0.11            1.30            1.73                3.68                5.64
Male                 0.14      1.15      0.08           -0.03            0.31                0.98                1.36
NonWhite            -0.24      0.79      0.15           -0.54            0.06                0.58                1.07
age_GBM_interaction -0.18      0.84      0.08           -0.33           -0.02                0.72                0.98

                     cmp to     z      p  -log2(p)
covariate                                         
age_centered10         0.00  7.50 <0.005     43.81
GBM                    0.00 13.95 <0.005    144.52
Male                   0.00  1.66   0.10      3.37
NonWhite               0.00 -1.54   0.12      3.02
age_GBM_interaction    0.00 -2.25   0.02      5.37
---
Concordance = 0.79
Partial AIC = 6654.04
log-likelihood ratio test = 589.81 on 5 df
-log2(p) of ll-ratio test = 413.55

In [31]:
# This is a likelihood ratio test that compares a reduced model without the age type interaction
# with a full model containing the interaction

reduced_columns = [
    "time",
    "event",
    "age_centered10",
    "GBM",
    "Male",
    "NonWhite"
]

reduced_cph = CoxPHFitter()

reduced_cph.fit(
    model_data[reduced_columns],
    duration_col="time",
    event_col="event",
    formula=(
        "age_centered10 + GBM + "
        "Male + NonWhite"
    )
)

likelihood_ratio = 2 * (cph.log_likelihood_ - reduced_cph.log_likelihood_)

interaction_p = stats.chi2.sf(likelihood_ratio,df=1)

print("Likelihood-ratio statistic:", f"{likelihood_ratio:.4f}")
print("Interaction p-value:", f"{interaction_p:.4f}")

Likelihood-ratio statistic: 5.0581
Interaction p-value: 0.0245


In [33]:
# The interaction shows that the age effect was statistically different between the groups
# even though age was associated with higher hazard in both groups

beta_age = cph.params_["age_centered10"]
beta_interaction = cph.params_["age_GBM_interaction"]

age_variance = cph.variance_matrix_.loc["age_centered10", "age_centered10"]

interaction_variance = cph.variance_matrix_.loc["age_GBM_interaction", "age_GBM_interaction"]

age_interaction_covariance = cph.variance_matrix_.loc["age_centered10","age_GBM_interaction"]

# LGG age effect
lgg_beta = beta_age
lgg_se = np.sqrt(age_variance)

# GBM age effect
gbm_beta = beta_age + beta_interaction

gbm_variance = (
    age_variance
    + interaction_variance
    + 2 * age_interaction_covariance
)

gbm_se = np.sqrt(gbm_variance)

for label, beta, se in [("LGG", lgg_beta, lgg_se),("GBM", gbm_beta, gbm_se)]:
    hazard_ratio = np.exp(beta)

    lower = np.exp(beta - 1.96 * se)
    upper = np.exp(beta + 1.96 * se)

    z_value = beta / se
    p_value = 2 * stats.norm.sf(abs(z_value))

    print(label)
    print("Hazard ratio:", f"{hazard_ratio:.3f}")
    print("95% CI:", f"{lower:.3f}", f"{upper:.3f}")
    print("p-value:", p_value)

LGG
Hazard ratio: 1.688
95% CI: 1.472 1.936
p-value: 6.468078017131333e-14
GBM
Hazard ratio: 1.415
95% CI: 1.317 1.521
p-value: 4.1836298805329935e-21


In [34]:
# The diagnostic test suggested that the proportional hazards assumption was violated for age, with \(p=.0007\).
# The other predictors did not show strong evidence of violation

proportional_hazards_test = proportional_hazard_test(
    cph,
    model_data[model_columns],
    time_transform="rank"
)

print(proportional_hazards_test.summary)

                     test_statistic         p   -log2(p)
GBM                        2.484701  0.114958   3.120819
Male                       2.034025  0.153813   2.700752
NonWhite                   2.463519  0.116517   3.101382
age_GBM_interaction        1.609098  0.204619   2.288990
age_centered10            11.604770  0.000658  10.570000
